# Practice: errors, logging and testing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-07-practice-errors-logging-and-testing.ipynb)

Companion to the [Errors, logging and testing](python-07-errors-logging-and-testing.ipynb) chapter. The exercises cover specific exceptions, retries, exception chaining, logging output and simple checks on your own functions.

## How to use this notebook

Work through the exercises in order. Each one has three parts:

1. **Question:** what to build, with the rules it must follow.
2. **Exercise cell:** replace the `pass` with your code. Keep the function name and parameters.
3. **Check cell:** run it after your code. Each line says `pass` when your function gives the expected result, and `todo` when it does not yet. Checks never stop the notebook, so you can run everything at any time.

The last cell prints your score. Use `Runtime -> Run all` to rerun everything after a change.

In [ ]:
_tally = {"passed": 0, "total": 0}

def check(label, make_actual, expected):
    """Run one check. It never raises, so the notebook always runs from top to bottom."""
    _tally["total"] += 1
    try:
        actual = make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

def raises(make_actual, error_type):
    """True when the call raises error_type, False when it raises anything else or nothing."""
    try:
        make_actual()
    except error_type:
        return True
    except Exception:
        return False
    return False

def score():
    print(f"\nScore: {_tally['passed']} of {_tally['total']} checks pass")

## Exercise 1: parse a percentage and raise a specific error

`InvalidPercentage` is given below. Write `parse_percentage(text)`. A text such as `"45%"` returns `0.45`. Text that is not a number followed by `%` raises `InvalidPercentage`.

*Hint: `float(text[:-1])` raises `ValueError` for `"x%"`. Catch that and raise your own error from it.*

In [ ]:
class InvalidPercentage(ValueError):
    """Raised for text that is not a percentage."""

def parse_percentage(text):
    # your code here
    pass

In [ ]:
check("parse_percentage('45%') == 0.45", lambda: parse_percentage("45%"), 0.45)
check("parse_percentage('100%') == 1.0", lambda: parse_percentage("100%"), 1.0)
check(
    "parse_percentage('x%') raises InvalidPercentage",
    lambda: raises(lambda: parse_percentage("x%"), InvalidPercentage),
    True,
)

<details>
<summary><b>Show solution</b></summary>

```python
class InvalidPercentage(ValueError):
    """Raised for text that is not a percentage."""

def parse_percentage(text):
    if not text.endswith("%"):
        raise InvalidPercentage(f"missing % sign: {text!r}")
    try:
        return float(text[:-1]) / 100
    except ValueError:
        raise InvalidPercentage(f"not a number: {text!r}") from None
```

</details>

## Exercise 2: retry until the call succeeds

Write `with_retries(fn, attempts)`. Call `fn()` up to `attempts` times. Return the first result that does not raise. If every attempt raises, raise the **last** error.

In [ ]:
def with_retries(fn, attempts):
    # your code here
    pass

In [ ]:
def fail_twice(state):
    state["calls"] += 1
    if state["calls"] <= 2:
        raise ConnectionError("temporary")
    return "ok"

def always_fail():
    raise ValueError("never works")

shared_state = {"calls": 0}
check(
    "with_retries returns the first success",
    lambda: with_retries(lambda: fail_twice(shared_state), 3),
    "ok",
)
check(
    "with_retries re-raises the last error",
    lambda: raises(lambda: with_retries(always_fail, 2), ValueError),
    True,
)

<details>
<summary><b>Show solution</b></summary>

```python
def with_retries(fn, attempts):
    last_error = None
    for _ in range(attempts):
        try:
            return fn()
        except Exception as error:
            last_error = error
    raise last_error
```

</details>

## Exercise 3: log a load summary

Write `log_load_summary(logger_name, loaded, rejected)`. Log one line at `INFO` level with the text `loaded <loaded> rows, rejected <rejected>`. Use `%d` placeholders, not an f-string.

In [ ]:
import logging

def log_load_summary(logger_name, loaded, rejected):
    # your code here
    pass

In [ ]:
import io

def capture_log(fn):
    """Run fn with a handler that writes to a string, and return what was logged."""
    stream = io.StringIO()
    logger = logging.getLogger("practice.load")
    logger.handlers.clear()
    handler = logging.StreamHandler(stream)
    handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    fn()
    return stream.getvalue().strip()

check(
    "log_load_summary logs one INFO line",
    lambda: capture_log(lambda: log_load_summary("practice.load", 120, 3)),
    "INFO loaded 120 rows, rejected 3",
)

<details>
<summary><b>Show solution</b></summary>

```python
import logging

def log_load_summary(logger_name, loaded, rejected):
    logging.getLogger(logger_name).info("loaded %d rows, rejected %d", loaded, rejected)
```

</details>

## Exercise 4: clamp a value and validate the bounds

Write `clamp(value, low, high)`. Return `value` limited to the range from `low` to `high`, inclusive. If `low` is greater than `high`, raise `ValueError`.

In [ ]:
def clamp(value, low, high):
    # your code here
    pass

In [ ]:
check("clamp(15, 0, 10) == 10", lambda: clamp(15, 0, 10), 10)
check("clamp(-3, 0, 10) == 0", lambda: clamp(-3, 0, 10), 0)
check("clamp(5, 0, 10) == 5", lambda: clamp(5, 0, 10), 5)
check("clamp(1, 5, 0) raises ValueError", lambda: raises(lambda: clamp(1, 5, 0), ValueError), True)

<details>
<summary><b>Show solution</b></summary>

```python
def clamp(value, low, high):
    if low > high:
        raise ValueError("low must not exceed high")
    return max(low, min(value, high))
```

</details>

## Exercise 5: median

Write `median(values)`. Return the middle value of the sorted list. For an even number of values, return the mean of the two middle values. An empty list raises `ValueError`.

In [ ]:
def median(values):
    # your code here
    pass

In [ ]:
check("median([3, 1, 2]) == 2", lambda: median([3, 1, 2]), 2)
check("median([4, 1, 3, 2]) == 2.5", lambda: median([4, 1, 3, 2]), 2.5)
check("median([]) raises ValueError", lambda: raises(lambda: median([]), ValueError), True)

<details>
<summary><b>Show solution</b></summary>

```python
def median(values):
    if not values:
        raise ValueError("median of an empty list")
    ordered = sorted(values)
    middle = len(ordered) // 2
    if len(ordered) % 2:
        return ordered[middle]
    return (ordered[middle - 1] + ordered[middle]) / 2
```

</details>

## Exercise 6: chain the original error

`LoadError` is given below. Write `load_int(text)`. It returns `int(text)`. If the text is not an integer, raise `LoadError` and keep the original `ValueError` as its cause.

In [ ]:
class LoadError(Exception):
    """A value could not be loaded."""

def load_int(text):
    # your code here
    pass

In [ ]:
def cause_is_value_error():
    try:
        load_int("x")
    except LoadError as error:
        return isinstance(error.__cause__, ValueError)
    return False

check("load_int('42') == 42", lambda: load_int("42"), 42)
check("load_int('x') keeps ValueError as the cause", cause_is_value_error, True)

<details>
<summary><b>Show solution</b></summary>

```python
class LoadError(Exception):
    """A value could not be loaded."""

def load_int(text):
    try:
        return int(text)
    except ValueError as error:
        raise LoadError(f"not an integer: {text!r}") from error
```

</details>

## Score

In [ ]:
score()